# Notebook 03 — Avaliação, Fronteira de Pareto e Processamento de Vídeo

## Objetivos

1. Avaliar modelos treinados com métricas de classificação e latência
2. Construir e analisar a fronteira de Pareto entre acurácia e latência
3. Implementar seleção de arquiteturas sob restrições de produção
4. Demonstrar keyframe extraction e temporal downsampling para vídeo
5. Comparar pipelines de augmentação (CPU vs GPU)

## Fundamentação Teórica

Conforme seção 10 do material (*Fronteira de Pareto e Seleção de Arquiteturas*):

> A seleção de arquitetura para sistemas de visão em produção é essencialmente
> um problema de otimização multi-objetivo na fronteira de Pareto entre acurácia,
> latência e custo. A fronteira define o conjunto de soluções onde não é possível
> melhorar um objetivo sem degradar outro.
>
> — Tan & Le (2019); Cai et al. (2020)

In [ ]:
# Imports
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import SimpleCNN, DepthwiseSeparableCNN, VisionBenchmarkModel
from src.data_preprocessing import DataPreprocessor
from src.training import create_dataloaders
from src.evaluation import (
    calculate_metrics,
    plot_pareto_frontier,
    plot_confusion_matrix,
    plot_training_history,
    generate_architecture_report,
)
from src.utils import load_model, set_seed, get_device, ensure_dirs

set_seed(42)
ensure_dirs('..')
device = get_device()
print(f'Dispositivo: {device}')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

## 1. Carregar Modelos Treinados e Dados

Carregamos os modelos salvos no notebook 02 e o dataset de benchmark.

In [ ]:
# Carregar dataset de imagens sintéticas
images, labels = DataPreprocessor.generate_synthetic_images(
    output_dir=None, n_samples=2000, img_size=64, n_classes=4, random_state=42,
)
_, val_loader = create_dataloaders(images, labels, batch_size=32, val_split=0.2, random_state=42)

# Carregar modelos
model_simple = SimpleCNN(n_classes=4)
model_mobile = DepthwiseSeparableCNN(n_classes=4)

try:
    ckpt_simple = load_model('../outputs/models/simple_cnn.pt')
    model_simple.load_state_dict(ckpt_simple['model_state_dict'])
    print(f'SimpleCNN carregado. Metadata: {ckpt_simple["metadata"]}')
except FileNotFoundError:
    print('SimpleCNN não encontrado — usando modelo não treinado')

try:
    ckpt_mobile = load_model('../outputs/models/mobilenet_style.pt')
    model_mobile.load_state_dict(ckpt_mobile['model_state_dict'])
    print(f'MobileNet-Style carregado. Metadata: {ckpt_mobile["metadata"]}')
except FileNotFoundError:
    print('MobileNet-Style não encontrado — usando modelo não treinado')

model_simple = model_simple.to(device).eval()
model_mobile = model_mobile.to(device).eval()

# Carregar dataset de benchmarks
dp = DataPreprocessor(random_state=42)
benchmark_path = '../data/raw/dataset.csv'
if not os.path.exists(benchmark_path):
    dp.generate_benchmark_dataset(benchmark_path)
df = dp.load_data(benchmark_path)
agg_df = dp.prepare_features(df)
print(f'\nBenchmark dataset: {len(agg_df)} configurações agregadas')

## 2. Avaliação: Métricas de Classificação

Calculamos acurácia, precisão, recall e F1-score para ambos os modelos.

In [ ]:
# Predições nos dados de validação
def get_predictions(model, loader, device):
    """Coleta predições e rótulos do DataLoader."""
    all_preds, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

class_names = ['Normal', 'Defeito', 'Tipo A', 'Anomalia']

# SimpleCNN
y_true_s, y_pred_s = get_predictions(model_simple, val_loader, device)
metrics_simple = calculate_metrics(y_true_s, y_pred_s, class_names=class_names)

# MobileNet-Style
y_true_m, y_pred_m = get_predictions(model_mobile, val_loader, device)
metrics_mobile = calculate_metrics(y_true_m, y_pred_m, class_names=class_names)

print('=== SimpleCNN ===')
print(f'Accuracy: {metrics_simple["accuracy"]:.4f}')
print(metrics_simple['classification_report'])

print('\n=== DepthwiseSeparableCNN (MobileNet-Style) ===')
print(f'Accuracy: {metrics_mobile["accuracy"]:.4f}')
print(metrics_mobile['classification_report'])

In [ ]:
# Matrizes de confusão lado a lado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

cm_simple = np.array(metrics_simple['confusion_matrix'])
cm_mobile = np.array(metrics_mobile['confusion_matrix'])

sns.heatmap(cm_simple, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax1)
ax1.set_title(f'SimpleCNN (Acc={metrics_simple["accuracy"]:.3f})')
ax1.set_xlabel('Predito')
ax1.set_ylabel('Real')

sns.heatmap(cm_mobile, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names, ax=ax2)
ax2.set_title(f'MobileNet-Style (Acc={metrics_mobile["accuracy"]:.3f})')
ax2.set_xlabel('Predito')
ax2.set_ylabel('Real')

plt.suptitle('Matrizes de Confusão — Comparação de Arquiteturas', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Fronteira de Pareto — Benchmark Dataset

Conforme seção 10 do material:

> A fronteira de Pareto define o conjunto de soluções onde não é possível
> melhorar um objetivo sem degradar outro — cada ponto na fronteira
> representa um trade-off ótimo para diferentes prioridades de design.
>
> — Tan & Le (2019)

In [ ]:
# Fronteira de Pareto para V100, resolução 224
v100_224 = agg_df[
    (agg_df['hardware'] == 'v100') & (agg_df['input_resolution'] == 224)
].copy()

fig = plot_pareto_frontier(
    v100_224,
    x_col='latency_mean_ms',
    y_col='top1_accuracy',
    color_col='model_family',
    size_col='params_millions',
    title='Fronteira de Pareto: Acurácia vs Latência\n(V100, resolução 224×224)',
    save_path='../outputs/figures/pareto_frontier_v100.png',
)
plt.show()

## 4. Seleção de Arquitetura sob Restrições

Conforme seção 10:

> Um sistema de detecção de defeitos industriais operando a 30 FPS em edge GPU
> (Jetson Orin Nano) requer latência < 33ms, restringindo opções a YOLOv8n/s
> ou EfficientNet-B0/B1.

In [ ]:
# Cenário 1: Edge deployment (latência < 5ms, acurácia > 70%)
print('=== Cenário 1: Edge Deployment (latência < 5ms, acurácia > 70%) ===')
report_edge = generate_architecture_report(
    v100_224, latency_constraint_ms=5.0, min_accuracy=70.0,
)
display_cols = ['model_name', 'model_family', 'params_millions', 'flops_giga',
                'top1_accuracy', 'latency_mean_ms', 'efficiency_score']
print(report_edge[display_cols].to_string(index=False))

# Cenário 2: Cloud deployment (latência < 15ms, acurácia > 80%)
print('\n=== Cenário 2: Cloud Deployment (latência < 15ms, acurácia > 80%) ===')
report_cloud = generate_architecture_report(
    v100_224, latency_constraint_ms=15.0, min_accuracy=80.0,
)
print(report_cloud[display_cols].to_string(index=False))

## 5. Keyframe Extraction

Conforme seção 6 do material (*Processamento de Vídeo*):

> Keyframe extraction identifica frames representativos que capturam mudanças
> significativas no conteúdo visual, permitindo processar apenas uma fração
> dos frames totais. Zhu et al. (2017) demonstraram que em vídeos típicos de
> vigilância, 80-90% dos frames são redundantes.

Demonstramos com frames sintéticos (sem necessidade de vídeo real).

In [ ]:
def generate_synthetic_video_frames(
    n_frames: int = 100,
    img_size: int = 128,
    n_scene_changes: int = 3,
    random_state: int = 42,
) -> np.ndarray:
    """
    Gera sequência sintética de frames simulando vídeo com mudanças de cena.
    
    Retorna: array (n_frames, img_size, img_size, 3), uint8
    """
    rng = np.random.RandomState(random_state)
    frames = np.zeros((n_frames, img_size, img_size, 3), dtype=np.uint8)
    
    # Definir cores de cena
    scene_colors = rng.randint(30, 200, size=(n_scene_changes + 1, 3))
    change_points = sorted(rng.choice(range(10, n_frames - 10), n_scene_changes, replace=False))
    change_points = [0] + list(change_points) + [n_frames]
    
    for seg_idx in range(len(change_points) - 1):
        start, end = change_points[seg_idx], change_points[seg_idx + 1]
        base_color = scene_colors[seg_idx]
        
        for f in range(start, end):
            # Frame base com ruído leve (frames consecutivos similares)
            noise = rng.randint(-5, 5, size=(img_size, img_size, 3))
            frame = np.clip(base_color + noise, 0, 255).astype(np.uint8)
            
            # Objeto em movimento
            cx = int(img_size * 0.2 + (f - start) * (img_size * 0.6) / max(end - start, 1))
            cy = img_size // 2
            r = 10
            yy, xx = np.ogrid[:img_size, :img_size]
            mask = ((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2
            frame[mask] = [255, 0, 0]  # Objeto vermelho
            
            frames[f] = frame
    
    return frames


def extract_keyframes_histogram(
    frames: np.ndarray,
    threshold: float = 0.3,
) -> tuple[list[int], list[float]]:
    """
    Extrai keyframes por diferença de histograma.
    
    Implementa abordagem baseada em diferença de histograma conforme
    discutido na seção 6 do material (Zhu et al., 2017).
    
    Args:
        frames: Array de frames (N, H, W, 3).
        threshold: Limiar de diferença para considerar keyframe.
    
    Returns:
        Tupla (keyframe_indices, diff_scores).
    """
    n_frames = len(frames)
    keyframe_indices = [0]  # Primeiro frame sempre é keyframe
    diff_scores = [0.0]
    
    for i in range(1, n_frames):
        # Calcular histogramas normalizados
        hist_prev = np.histogram(frames[i-1], bins=64, range=(0, 255))[0].astype(float)
        hist_curr = np.histogram(frames[i], bins=64, range=(0, 255))[0].astype(float)
        
        hist_prev /= hist_prev.sum() + 1e-10
        hist_curr /= hist_curr.sum() + 1e-10
        
        # Chi-squared distance
        diff = np.sum((hist_prev - hist_curr) ** 2 / (hist_prev + hist_curr + 1e-10))
        diff_scores.append(diff)
        
        if diff > threshold:
            keyframe_indices.append(i)
    
    return keyframe_indices, diff_scores


# Gerar vídeo sintético
frames = generate_synthetic_video_frames(n_frames=100, n_scene_changes=3)
print(f'Frames gerados: {frames.shape}')

# Extrair keyframes
keyframe_idx, diff_scores = extract_keyframes_histogram(frames, threshold=0.3)
print(f'Keyframes extraídos: {len(keyframe_idx)} de {len(frames)} frames')
print(f'Redução: {(1 - len(keyframe_idx)/len(frames))*100:.1f}% dos frames descartados')
print(f'Índices dos keyframes: {keyframe_idx}')

In [ ]:
# Visualizar diferenças e keyframes
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Score de diferença entre frames
axes[0].plot(diff_scores, color='#2196F3', alpha=0.7)
axes[0].axhline(y=0.3, color='r', linestyle='--', label='Threshold=0.3')
for kf in keyframe_idx:
    axes[0].axvline(x=kf, color='green', alpha=0.3, linewidth=2)
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Diferença de Histograma')
axes[0].set_title('Detecção de Keyframes por Diferença de Histograma (Zhu et al., 2017)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mostrar keyframes
n_show = min(len(keyframe_idx), 6)
for i in range(n_show):
    ax_kf = fig.add_axes([0.05 + i * 0.16, 0.02, 0.14, 0.35])
    ax_kf.imshow(frames[keyframe_idx[i]])
    ax_kf.set_title(f'Frame {keyframe_idx[i]}', fontsize=9)
    ax_kf.axis('off')

axes[1].axis('off')
axes[1].set_title('Keyframes Selecionados', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/keyframe_extraction.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Temporal Downsampling

Conforme seção 6 do material:

> Temporal downsampling aplica amostragem uniforme ou adaptativa sobre a
> sequência de frames. Wu et al. (2019) demonstraram redução de 50% no custo
> com degradação < 1% em acurácia.

In [ ]:
# Comparação de estratégias de temporal downsampling
from src.utils import BenchmarkTimer

def simulate_video_processing(frames, process_fn, sample_rate=1):
    """Simula processamento de vídeo com downsampling."""
    selected = frames[::sample_rate]
    timer = BenchmarkTimer()
    with timer.measure():
        for frame in selected:
            process_fn(frame)
    return len(selected), timer.measurements[0]

# Função de processamento simulada (redimensionar + normalizar)
def process_frame(frame):
    """Simula processamento de um frame."""
    resized = frame[::2, ::2, :]  # Downsample simples
    normalized = resized.astype(np.float32) / 255.0
    return normalized

# Benchmark com diferentes taxas de amostragem
sample_rates = [1, 2, 3, 5, 10, 15]
results = []

for rate in sample_rates:
    n_processed, time_ms = simulate_video_processing(frames, process_frame, rate)
    results.append({
        'sample_rate': rate,
        'frames_processed': n_processed,
        'time_ms': time_ms,
        'fps_equivalent': n_processed / (time_ms / 1000) if time_ms > 0 else 0,
        'coverage': n_processed / len(frames) * 100,
    })

results_df = pd.DataFrame(results)
print('=== Temporal Downsampling ===')
print(results_df.to_string(index=False))

# Visualizar trade-off
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(results_df['sample_rate'], results_df['time_ms'], 'o-', color='#F44336')
ax1.set_xlabel('Taxa de Amostragem (1 a cada N frames)')
ax1.set_ylabel('Tempo de Processamento (ms)')
ax1.set_title('Custo vs Taxa de Amostragem')
ax1.grid(True, alpha=0.3)

ax2.plot(results_df['sample_rate'], results_df['coverage'], 's-', color='#4CAF50')
ax2.set_xlabel('Taxa de Amostragem')
ax2.set_ylabel('Cobertura (%)')
ax2.set_title('Cobertura vs Taxa de Amostragem')
ax2.grid(True, alpha=0.3)

plt.suptitle('Temporal Downsampling — Trade-off Custo vs Cobertura (Wu et al., 2019)',
             fontsize=13)
plt.tight_layout()
plt.show()

## 7. Comparação de Pipelines de Augmentação

Conforme seção 8 do material (*Augmentações Otimizadas*):

> Operações de augmentação podem consumir tempo comparável à própria inferência.
> DALI executa diretamente na GPU, alcançando speedups de 2-4x.

Comparamos throughput de pipelines CPU (torchvision) simulado.

In [ ]:
import torchvision.transforms as T

# Pipeline de augmentação CPU (torchvision)
transform_cpu = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Pipeline simplificado (tensor-only, sem PIL)
transform_fast = T.Compose([
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Benchmark
n_images = 200
test_images = torch.from_numpy(images[:n_images])  # Já em formato (N, C, H, W)

timer = BenchmarkTimer()

# CPU pipeline (com PIL)
def run_cpu_pipeline():
    for img in test_images:
        # Converter para uint8 para ToPILImage
        img_uint8 = (img * 255).byte()
        transform_cpu(img_uint8)

stats_cpu = timer.run(run_cpu_pipeline, n_warmup=1, n_runs=3)

# Fast pipeline (tensor-only)
def run_fast_pipeline():
    for img in test_images:
        transform_fast(img)

stats_fast = timer.run(run_fast_pipeline, n_warmup=1, n_runs=3)

print('=== Benchmark de Pipelines de Augmentação ===')
print(f'CPU (PIL + transforms):  {stats_cpu["mean_ms"]:.1f} ms para {n_images} imagens')
print(f'  Throughput: {n_images / (stats_cpu["mean_ms"] / 1000):.1f} imgs/s')
print(f'\nFast (tensor-only):      {stats_fast["mean_ms"]:.1f} ms para {n_images} imagens')
print(f'  Throughput: {n_images / (stats_fast["mean_ms"] / 1000):.1f} imgs/s')

# Tabela comparativa com dados da seção 8 (Tabela 4)
print('\n=== Referência: Tabela 4 do Material da Aula ===')
ref_data = pd.DataFrame([
    {'Framework': 'PIL + transforms', 'Execução GPU': 'Não', 'Decode GPU': 'Não', 'Throughput (img/s)': 850},
    {'Framework': 'torchvision.transforms.v2', 'Execução GPU': 'Parcial', 'Decode GPU': 'Não', 'Throughput (img/s)': 1200},
    {'Framework': 'Albumentations', 'Execução GPU': 'Não', 'Decode GPU': 'Não', 'Throughput (img/s)': 1100},
    {'Framework': 'NVIDIA DALI', 'Execução GPU': 'Sim', 'Decode GPU': 'Sim', 'Throughput (img/s)': 3500},
])
print(ref_data.to_string(index=False))

## 8. Análise de Variantes YOLO

Conforme Tabela 3 do material, a família YOLOv8 oferece variantes
escalonadas para diferentes restrições de latência (Jocher et al., 2023).

In [ ]:
# Dados da Tabela 3 do material
yolo_data = pd.DataFrame([
    {'Variante': 'YOLOv8n', 'Parâmetros (M)': 3.2, 'FLOPs (G)': 8.7,
     'mAP@0.5:0.95': 37.3, 'Latência V100 (ms)': 1.8},
    {'Variante': 'YOLOv8s', 'Parâmetros (M)': 11.2, 'FLOPs (G)': 28.6,
     'mAP@0.5:0.95': 44.9, 'Latência V100 (ms)': 2.4},
    {'Variante': 'YOLOv8m', 'Parâmetros (M)': 25.9, 'FLOPs (G)': 78.9,
     'mAP@0.5:0.95': 50.2, 'Latência V100 (ms)': 4.1},
    {'Variante': 'YOLOv8l', 'Parâmetros (M)': 43.7, 'FLOPs (G)': 165.2,
     'mAP@0.5:0.95': 52.9, 'Latência V100 (ms)': 6.8},
    {'Variante': 'YOLOv8x', 'Parâmetros (M)': 68.2, 'FLOPs (G)': 257.8,
     'mAP@0.5:0.95': 53.9, 'Latência V100 (ms)': 9.3},
])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# mAP vs Latência
axes[0].plot(yolo_data['Latência V100 (ms)'], yolo_data['mAP@0.5:0.95'],
             'o-', color='#FF5722', markersize=10)
for _, row in yolo_data.iterrows():
    axes[0].annotate(row['Variante'],
                     (row['Latência V100 (ms)'], row['mAP@0.5:0.95']),
                     textcoords='offset points', xytext=(5, 5), fontsize=9)
axes[0].set_xlabel('Latência V100 (ms)')
axes[0].set_ylabel('mAP@0.5:0.95')
axes[0].set_title('mAP vs Latência')
axes[0].grid(True, alpha=0.3)

# FLOPs vs mAP
axes[1].plot(yolo_data['FLOPs (G)'], yolo_data['mAP@0.5:0.95'],
             's-', color='#2196F3', markersize=10)
for _, row in yolo_data.iterrows():
    axes[1].annotate(row['Variante'],
                     (row['FLOPs (G)'], row['mAP@0.5:0.95']),
                     textcoords='offset points', xytext=(5, 5), fontsize=9)
axes[1].set_xlabel('FLOPs (GFLOPs)')
axes[1].set_ylabel('mAP@0.5:0.95')
axes[1].set_title('mAP vs FLOPs')
axes[1].grid(True, alpha=0.3)

# Parâmetros vs mAP
axes[2].bar(yolo_data['Variante'], yolo_data['Parâmetros (M)'], color='#4CAF50', alpha=0.7)
ax2 = axes[2].twinx()
ax2.plot(yolo_data['Variante'], yolo_data['mAP@0.5:0.95'], 'ro-', markersize=8)
axes[2].set_ylabel('Parâmetros (M)', color='#4CAF50')
ax2.set_ylabel('mAP@0.5:0.95', color='red')
axes[2].set_title('Parâmetros e mAP por Variante')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Família YOLOv8 — Trade-offs (Jocher et al., 2023)', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/yolo_variants.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Análise Final: Recomendações de Deployment

Consolidamos os aprendizados em recomendações práticas conforme
a metodologia da seção 10 (fronteira de Pareto).

In [ ]:
# Tabela de recomendações por cenário
recommendations = pd.DataFrame([
    {'Cenário': 'Edge (Jetson Nano)', 'Latência Max': '< 33ms',
     'Recomendação': 'YOLOv8n / MobileNetV2', 'Justificativa': 'Menor FLOPs, cabe em edge GPU'},
    {'Cenário': 'Edge (Jetson Orin)', 'Latência Max': '< 10ms',
     'Recomendação': 'EfficientNet-B0 / YOLOv8s', 'Justificativa': 'Bom trade-off acurácia-latência'},
    {'Cenário': 'Cloud (T4 GPU)', 'Latência Max': '< 50ms',
     'Recomendação': 'EfficientNetV2-S / Swin-Tiny', 'Justificativa': 'Alta acurácia com latência aceitável'},
    {'Cenário': 'Cloud (V100/A100)', 'Latência Max': '< 100ms',
     'Recomendação': 'DeiT-Base / ConvNeXt-Small', 'Justificativa': 'Máxima acurácia, hardware permite'},
    {'Cenário': 'Vídeo 30 FPS', 'Latência Max': '< 33ms',
     'Recomendação': 'YOLOv8n + keyframe extraction', 'Justificativa': 'Combinação de modelo rápido + redução temporal'},
])

print('=== Recomendações de Arquitetura por Cenário de Deployment ===')
print(recommendations.to_string(index=False))

## Resumo

Neste notebook:

1. **Avaliamos modelos** treinados com métricas de classificação e matrizes de confusão
2. **Construímos a fronteira de Pareto** — identificando trade-offs ótimos entre acurácia e latência (Tan & Le, 2019)
3. **Selecionamos arquiteturas** sob restrições específicas de latência para cenários edge e cloud
4. **Implementamos keyframe extraction** por diferença de histograma, reduzindo frames processados em 80-90% (Zhu et al., 2017)
5. **Demonstramos temporal downsampling** — trade-off entre custo e cobertura (Wu et al., 2019)
6. **Comparamos pipelines de augmentação** — DALI oferece ~4x speedup sobre PIL (NVIDIA, 2023)
7. **Analisamos variantes YOLOv8** — nano a extra-large para diferentes SLAs (Jocher et al., 2023)

**Conclusão:** A seleção de arquitetura em produção deve ser guiada pela fronteira de Pareto,
considerando restrições de hardware, latência e orçamento computacional.

**Próxima aula:** Otimização de Modelos com Pruning e Quantização.